In [9]:
import pandas as pd
import numpy as np
import pickle
import joblib
import re
import string
from tqdm.notebook import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from underthesea import word_tokenize

# 1. Đọc stopwords
try:
    with open("vietnamese-stopwords-dash.txt", 'r', encoding='utf-8') as f:
        stopwords = f.read().splitlines()
except:
    stopwords = []

# 2. Hàm tiền xử lý độc lập
def preprocess_text(text):
    if not text or not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text, format="text").split()
    return " ".join([w for w in tokens if w not in stopwords])

def normalize_scores(scores):
    if len(scores) == 0: return scores
    min_s, max_s = np.min(scores), np.max(scores)
    if max_s - min_s == 0: return np.zeros_like(scores)
    return (scores - min_s) / (max_s - min_s)

print("✅ Đã khởi tạo các hàm tiện ích thành công!")


✅ Đã khởi tạo các hàm tiện ích thành công!


In [10]:
print("🚀 Đang nạp dữ liệu và Models (Chỉ load TF-IDF & BGE-M3)...")

# 1. Đọc dữ liệu công việc và Query
df_jobs = pd.read_excel("data/df_processed.xlsx", engine='openpyxl')
if 'id' not in df_jobs.columns:
    df_jobs['id'] = df_jobs.index
    
df_queries = pd.read_json("eval/queries_natural_200.json")

# 2. Load TF-IDF (Bản Upgrade - Overall)
tfidf_model = joblib.load("tfidf_model_vi_basic.joblib")
tfidf_matrix = joblib.load("tfidf_matrix_basic.pkl")

# 3. Load BGE-M3 (Bản Upgrade - Overall)
# Lưu ý: Cập nhật lại đường dẫn thư mục model bge_m3 của bạn nếu cần
bge_model = SentenceTransformer("bge_m3_model_vn_basic", device="cpu")
bge_matrix = np.load("job_title_embeddings_bge_m3_vn_basic.npy")

print(f"✅ Đã nạp xong {len(df_jobs)} jobs và {len(df_queries)} queries.")
print("✅ Ma trận TF-IDF và BGE-M3 đã sẵn sàng trên RAM!")

🚀 Đang nạp dữ liệu và Models (Chỉ load TF-IDF & BGE-M3)...


c:\Users\Quyen\anaconda3\envs\job_recsys\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Quyen\anaconda3\envs\job_recsys\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ Đã nạp xong 6473 jobs và 205 queries.
✅ Ma trận TF-IDF và BGE-M3 đã sẵn sàng trên RAM!


In [11]:
def standalone_ensemble_search(query, w_bge, w_tfidf, top_k=20):
    query_str = preprocess_text(query)
    indices = df_jobs.index

    # 1. Tính TF-IDF
    vec_tfidf = tfidf_model.transform([query_str])
    raw_tfidf = cosine_similarity(vec_tfidf, tfidf_matrix).flatten()
    sub_tfidf = raw_tfidf[indices] if max(indices) < len(raw_tfidf) else np.zeros(len(indices))

    # 2. Tính BGE-M3
    vec_bge = bge_model.encode([query_str], normalize_embeddings=True)
    raw_bge = cosine_similarity(vec_bge, bge_matrix).flatten()
    sub_bge = raw_bge[indices] if max(indices) < len(raw_bge) else np.zeros(len(indices))

    # 3. Tổng hợp điểm
    norm_tfidf = normalize_scores(sub_tfidf)
    norm_bge = normalize_scores(sub_bge)
    final_scores = (w_bge * norm_bge) + (w_tfidf * norm_tfidf)

    # 4. Trả về kết quả
    df_results = df_jobs.copy()
    df_results['similarity_score'] = final_scores
    
    # Trả về trực tiếp list ID của top K
    return df_results.nlargest(top_k, 'similarity_score')['id'].tolist()

In [12]:
# Mốc trọng số BGE-M3 từ 0.0 đến 1.0
bge_weights = np.round(np.arange(0.0, 1.1, 0.1), 1)
retrieval_results = {}

print("🔥 Bắt đầu chạy Ablation Study...")

for w_bge in bge_weights:
    w_tfidf = round(1.0 - w_bge, 1)
    print(f"\n🔄 Hệ số: BGE-M3 = {w_bge:.1f} | TF-IDF = {w_tfidf:.1f}")
    
    current_weight_results = {}
    
    for q_idx, row in tqdm(df_queries.iterrows(), total=len(df_queries)):
        query_text = row['query']
        
        # Gọi hàm search vừa tạo
        retrieved_ids = standalone_ensemble_search(
            query=query_text, 
            w_bge=w_bge, 
            w_tfidf=w_tfidf, 
            top_k=20
        )
        current_weight_results[q_idx] = retrieved_ids
        
    retrieval_results[w_bge] = current_weight_results

# Lưu kết quả xuống file để xài cho bước chấm điểm (Ground Truth)
with open("semlex_retrieval_results_title.pkl", "wb") as f:
    pickle.dump(retrieval_results, f)

print("\n🎉 XONG! Đã lưu kết quả của toàn bộ các mốc trọng số.")

🔥 Bắt đầu chạy Ablation Study...

🔄 Hệ số: BGE-M3 = 0.0 | TF-IDF = 1.0


  0%|          | 0/205 [00:00<?, ?it/s]


🔄 Hệ số: BGE-M3 = 0.1 | TF-IDF = 0.9


  0%|          | 0/205 [00:00<?, ?it/s]


🔄 Hệ số: BGE-M3 = 0.2 | TF-IDF = 0.8


...


🔄 Hệ số: BGE-M3 = 0.9 | TF-IDF = 0.1


  0%|          | 0/205 [00:00<?, ?it/s]


🔄 Hệ số: BGE-M3 = 1.0 | TF-IDF = 0.0


  0%|          | 0/205 [00:00<?, ?it/s]


🎉 XONG! Đã lưu kết quả của toàn bộ các mốc trọng số.
